In [1]:
!pip install -q librosa xgboost

import glob
import os
import time

import librosa
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
from scipy.signal import butter, filtfilt

DATASET_PATH = 'ICBHI_final_database'
SPLIT_FILE = 'ICBHI_final_database/ICBHI_challenge_train_test.txt'
TARGET_SAMPLING_RATE = 16000


# Remove sounds from skin friction or medical equipment beeps (100-4000 Hz preserves fine crackles)
def bandpassFilter(signal):
    nyquistFrequency = 0.5 * TARGET_SAMPLING_RATE
    b, a = butter(5, [100.0 / nyquistFrequency, 4000.0 / nyquistFrequency], btype='band')
    return filtfilt(b, a, signal)


# Breath type from crackle and wheeze
def getBreathType(crackle, wheeze):
    match [int(crackle), int(wheeze)]:
        case [0, 0]:
            return 0  # Normal
        case [1, 0]:
            return 1  # Crackle
        case [0, 1]:
            return 2  # Wheeze
        case [1, 1]:
            return 3  # Both


def getSplitMap():
    splitMap = {}

    # Return early if the split file path is empty or doesn't exist
    if not SPLIT_FILE or not os.path.exists(SPLIT_FILE):
        return splitMap

    with open(SPLIT_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                splitMap[parts[0]] = parts[1]

    return splitMap


def getMetaDataFromFileName(parts):
    patientId = int(parts[0])
    chestLocation = parts[2]
    acqMode = parts[3]
    device = parts[4]

    return patientId, chestLocation, acqMode, device


def parseDataset():
    split_map = getSplitMap()

    # Get all the txt files
    txtFiles = sorted(glob.glob(os.path.join(DATASET_PATH, "*.txt")))
    records = []
    processed = []

    for path in txtFiles:
        baseName = os.path.splitext(os.path.basename(path))[0]
        wavPath = os.path.join(DATASET_PATH, baseName + ".wav")
        parts = baseName.split("_")

        # Skip if .wav doesn't exist or filename isn't correctly formatted
        if not os.path.exists(wavPath) or len(parts) != 5:
            continue

        patientId, chestLocation, acqMode, device = getMetaDataFromFileName(parts)
        splitLabel = split_map.get(baseName, "unknown")

        # Load the audio and pass it through bandpass filter
        full_audio, _ = librosa.load(wavPath, sr=TARGET_SAMPLING_RATE)
        full_audio = bandpassFilter(full_audio)

        # Get annotations
        annotations = pd.read_csv(path, sep="\t", names=["start", "end", "crackle", "wheeze"])

        for idx, row in annotations.iterrows():
            start_sample = int(row["start"] * TARGET_SAMPLING_RATE)
            end_sample = int(row["end"] * TARGET_SAMPLING_RATE)

            # Slice the active parts of the audio
            cycle_audio = full_audio[start_sample:end_sample]
            if len(cycle_audio) == 0:
                continue

            # Normalise the volume
            if np.std(cycle_audio) > 0:
                cycle_audio = (cycle_audio - np.mean(cycle_audio)) / np.std(cycle_audio)

            records.append({
                "filename": baseName,
                "patient_id": patientId,
                "cycle_idx": idx,
                "duration_sec": row["end"] - row["start"],
                "chest_loc": chestLocation,
                "device": device,
                "split": splitLabel,
                "label": getBreathType(row["crackle"], row["wheeze"])
            })
            processed.append(cycle_audio.astype(np.float32))

    df = pd.DataFrame(records)
    return df, processed


df_meta, X_audio = parseDataset()

print(f"Total cycles extracted: {len(df_meta)} (Expected: 6898)")
print(f"Variable-length waveforms stored: {len(X_audio)}")
print("\nClass Distribution:")
print(df_meta["label"].value_counts().sort_index())
print("\nTrain/Test Split Distribution:")
print(df_meta["split"].value_counts())

Total cycles extracted: 6898 (Expected: 6898)
Variable-length waveforms stored: 6898

Class Distribution:
label
0    3642
1    1864
2     886
3     506
Name: count, dtype: int64

Train/Test Split Distribution:
split
train      4131
test       2756
unknown      11
Name: count, dtype: int64


In [3]:
def extractFeatures(audio):
    features = []
    n_fft = 512
    hop_length = 128

    for cycle in audio:
        min_samples = n_fft + 8 * hop_length
        
        if len(cycle) < min_samples:
            cycle = np.pad(cycle, (0, min_samples - len(cycle)), mode="edge")

        mfcc = librosa.feature.mfcc(
            y=cycle, sr=TARGET_SAMPLING_RATE, n_mfcc=20, n_fft=n_fft, hop_length=hop_length, fmin=100.0, fmax=4000.0
        )
        mfccDelta = librosa.feature.delta(mfcc)

        mfccMean = np.mean(mfcc, axis=1)
        mfccStd = np.std(mfcc, axis=1)
        deltaMean = np.mean(mfccDelta, axis=1)
        deltaStd = np.std(mfccDelta, axis=1)

        # Chroma
        chroma = librosa.feature.chroma_stft(
            y=cycle, sr=TARGET_SAMPLING_RATE, n_fft=n_fft, hop_length=hop_length
        )
        chromaMean = np.mean(chroma, axis=1)
        chromaStd = np.std(chroma, axis=1)

        # Spectral Features
        centroid = librosa.feature.spectral_centroid(
            y=cycle, sr=TARGET_SAMPLING_RATE, n_fft=n_fft, hop_length=hop_length
        )
        bandwidth = librosa.feature.spectral_bandwidth(
            y=cycle, sr=TARGET_SAMPLING_RATE, n_fft=n_fft, hop_length=hop_length
        )
        rolloff = librosa.feature.spectral_rolloff(
            y=cycle, sr=TARGET_SAMPLING_RATE, n_fft=n_fft, hop_length=hop_length
        )
        # fmin=100.0 with 5 bands covers 100 Hz -> 3200 Hz (stays strictly below 8000 Hz Nyquist)
        contrast = librosa.feature.spectral_contrast(
            y=cycle, sr=TARGET_SAMPLING_RATE, n_fft=n_fft, hop_length=hop_length, fmin=100.0, n_bands=5
        )
        zcr = librosa.feature.zero_crossing_rate(
            y=cycle, frame_length=n_fft, hop_length=hop_length
        )

        contrastMean = np.mean(contrast, axis=1)
        contrastStd = np.std(contrast, axis=1)

        spectral_stats = np.array([
            np.mean(centroid), np.std(centroid),
            np.mean(bandwidth), np.std(bandwidth),
            np.mean(rolloff), np.std(rolloff),
            np.mean(zcr), np.std(zcr)
        ])

        features.append(
            np.concatenate([
                mfccMean, mfccStd,
                deltaMean, deltaStd,
                chromaMean, chromaStd,
                spectral_stats,
                contrastMean, contrastStd
            ])
        )

    return np.array(features, dtype=np.float32)


X_features = extractFeatures(X_audio)
print(f"Extracted feature matrix shape: {X_features.shape}")

Extracted feature matrix shape: (6898, 124)


In [7]:
# Assign any unknown files by patient_id so patient recordings never leak across train/test
unknown = df_meta.loc[df_meta["split"] == "unknown", "patient_id"].unique()
for patientId in unknown:
    existing = df_meta.loc[(df_meta["patient_id"] == patientId) & (df_meta["split"] != "unknown"), "split"].unique()
    assigned = existing[0] if len(existing) > 0 else "train"
    df_meta.loc[(df_meta["patient_id"] == patientId) & (df_meta["split"] == "unknown"), "split"] = assigned

# Use snake_case consistently so Cells 5, 6, and 7 work without NameErrors
train_mask = (df_meta["split"] == "train").values
test_mask = (df_meta["split"] == "test").values

# Slice raw features and labels (device-wise scaling happens next using train_mask only)
X_train = X_features[train_mask]
y_train = df_meta.loc[train_mask, "label"].values

X_test = X_features[test_mask]
y_test = df_meta.loc[test_mask, "label"].values

train_patients = df_meta.loc[train_mask, "patient_id"].unique()
test_patients = df_meta.loc[test_mask, "patient_id"].unique()

print(f"X_train: {X_train.shape}, y_train: {y_train.shape} ({len(train_patients)} patients)")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape} ({len(test_patients)} patients)")

X_train: (4142, 124), y_train: (4142,) (79 patients)
X_test:  (2756, 124),  y_test:  (2756,) (49 patients)


In [8]:
# Scale the data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
import time
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
from xgboost import XGBClassifier

X_dev_norm = np.zeros_like(X_features, dtype=np.float32)

# Fallback global scaler in case a test recording comes from a device unseen in train
globalScaler = StandardScaler().fit(X_features[train_mask])

# Normalise per recording device to mitigate hardware frequency-response differences
for device in df_meta["device"].unique():
    dev_train_mask = (df_meta["device"] == device).values & train_mask
    dev_test_mask  = (df_meta["device"] == device).values & test_mask

    if np.any(dev_train_mask):
        deviceScaler = StandardScaler()
        X_dev_norm[dev_train_mask] = deviceScaler.fit_transform(X_features[dev_train_mask])
        if np.any(dev_test_mask):
            X_dev_norm[dev_test_mask] = deviceScaler.transform(X_features[dev_test_mask])
    elif np.any(dev_test_mask):
        X_dev_norm[dev_test_mask] = globalScaler.transform(X_features[dev_test_mask])

X_train_dev = X_dev_norm[train_mask]
X_test_dev = X_dev_norm[test_mask]

raw_weights = compute_sample_weight("balanced", y_train)
smooth_weights = np.sqrt(raw_weights)

# Define our models
models = {
    "SVM": SVC(kernel="rbf", C=2.0, gamma="scale", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=18, min_samples_leaf=2, random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(learning_rate=0.04, max_iter=250, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.04, max_depth=5, subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0, tree_method="hist", random_state=42, n_jobs=-1),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=15, metric="cosine", weights="distance", n_jobs=-1),
    "Linear SVM": LinearSVC(C=0.05, max_iter=5000, random_state=42),
    "Logistic Regression": LogisticRegression(C=0.2, max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=42),
}


# Function to calculate the ICBHI score
def icbhiScore(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    sp = cm[0, 0] / cm[0].sum()
    se = (cm[1, 1] + cm[2, 2] + cm[3, 3]) / cm[1:].sum()
    return se, sp, (se + sp) / 2.0


results = []
predictions = {}

# Loop through our models
for name, model in models.items():
    start = time.time()

    # Pass smoothed weights for all models except k-nearest neighbors
    if name == "K-Nearest Neighbors":
        model.fit(X_train_dev, y_train)
    else:
        model.fit(X_train_dev, y_train, sample_weight=smooth_weights)

    mins, secs = divmod(int(time.time() - start), 60)

    # Pass test data to trained model
    y_pred = model.predict(X_test_dev)
    predictions[name] = y_pred

    # Get metrics from predictions
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)

    # Calculate the ICBHI score
    se, sp, icbhi = icbhiScore(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ICBHI Score": icbhi,
        "Se": se,
        "Sp": sp,
        "Precision (Macro)": prec,
        "Recall (Macro)": rec,
        "F1-Score (Macro)": f1,
        "Training Time": f"{mins}m {secs}s",
    })

results_df = pd.DataFrame(results).sort_values(by="ICBHI Score", ascending=False)

print("============================== Classical ML Model Comparison ==============================\n")
print(results_df.to_string(index=False))

/Users/henryburbridge/Desktop/AI-module-ARU/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [11]:
# Convert audio to an image spectogram 
def convertToSpectogram(X_waveforms, img_size=(128, 256)):
    specs = []
    for cycle in X_waveforms:
        if len(cycle) < 512:
            cycle = np.pad(cycle, (0, 512 - len(cycle)), mode="constant")

        mel = librosa.feature.melspectrogram(
            y=cycle,
            sr=TARGET_SAMPLING_RATE,
            n_fft=512,
            hop_length=128,
            n_mels=128,
            fmin=100.0,
            fmax=4000.0,
        )
        log_mel = librosa.power_to_db(mel, ref=np.max, top_db=80.0)
        delta1 = librosa.feature.delta(log_mel)
        delta2 = librosa.feature.delta(log_mel, order=2)

        # Stack into 3 distinct channels: (128, T, 3)
        img_3ch = np.stack([log_mel, delta1, delta2], axis=-1)

        # Resize to fixed (128, 256) preserving temporal resolution for crackles
        img_resized = tf.image.resize(img_3ch, img_size).numpy()
        specs.append(img_resized.astype(np.float32))

    return np.array(specs, dtype=np.float32)


X_specs = convertToSpectogram(X_audio, img_size=(128, 256))
    
X_specs = convertToSpectogram(X_audio)
print(f"Keras spectrogram tensor shape: {X_specs.shape}")  # (6898, 128, 188, 3)

X_train_cnn = X_specs[train_mask]
y_train_cnn = df_meta.loc[train_mask, "label"].values

X_test_cnn = X_specs[test_mask]
y_test_cnn = df_meta.loc[test_mask, "label"].values

2026-11-16 16:13:06.865453: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Pro
2026-11-16 16:13:06.865504: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-11-16 16:13:06.865521: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-11-16 16:13:06.865561: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-16 16:13:06.865591: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Keras spectrogram tensor shape: (6898, 128, 256, 3)


In [16]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainRowNumbers = np.where(train_mask)[0]

sub_train_idx, val_idx = next(
    gss.split(
        X_specs[trainRowNumbers],
        df_meta.loc[train_mask, "label"].values,
        groups=df_meta.loc[train_mask, "patient_id"].values,
    )
)

# Row indices in df_meta / X_specs for each split
sub_train_rows = trainRowNumbers[sub_train_idx]
val_rows       = trainRowNumbers[val_idx]
test_rows      = np.where(test_mask)[0]

X_tr_raw   = X_specs[sub_train_rows]
y_tr_cnn   = df_meta.iloc[sub_train_rows]["label"].values
dev_tr     = df_meta.iloc[sub_train_rows]["device"].values

X_val_raw  = X_specs[val_rows]
y_val_cnn  = df_meta.iloc[val_rows]["label"].values
dev_val    = df_meta.iloc[val_rows]["device"].values

X_test_raw = X_specs[test_rows]
y_test_cnn = df_meta.iloc[test_rows]["label"].values
dev_test   = df_meta.iloc[test_rows]["device"].values

# Compute global sub-train stats per (frequency_bin, channel) as fallback
# Shape: (1, H, 1, C) -> preserves temporal contrast while equalizing frequency response
global_mean = X_tr_raw.mean(axis=(0, 2), keepdims=True)
global_std  = X_tr_raw.std(axis=(0, 2), keepdims=True) + 1e-6

# Compute per-device stats using ONLY sub-train samples
device_stats = {}
for dev in np.unique(dev_tr):
    mask = (dev_tr == dev)
    if np.sum(mask) > 5:  # Guard against tiny device subsets
        d_mean = X_tr_raw[mask].mean(axis=(0, 2), keepdims=True)
        d_std  = X_tr_raw[mask].std(axis=(0, 2), keepdims=True) + 1e-6
        device_stats[dev] = (d_mean, d_std)


def normalize_by_device_to_255(X, devices):
    X_norm = np.empty_like(X, dtype=np.float32)
    for dev in np.unique(devices):
        mask = (devices == dev)
        mean, std = device_stats.get(dev, (global_mean, global_std))
        z = np.clip((X[mask] - mean) / std, -3.5, 3.5)
        X_norm[mask] = ((z + 3.5) / 7.0) * 255.0
    return X_norm


X_tr_cnn   = normalize_by_device_to_255(X_tr_raw, dev_tr)
X_val_cnn  = normalize_by_device_to_255(X_val_raw, dev_val)
X_test_cnn = normalize_by_device_to_255(X_test_raw, dev_test)


def maskRandomSlice(image, axis):
    if tf.random.uniform(()) >= 0.5:
        return image

    fillValue = 127.5
    shape = tf.shape(image)
    axis_len = shape[axis]
    max_w = 12 if axis == 1 else 16
    width = tf.random.uniform((), minval=4, maxval=max_w, dtype=tf.int32)
    start = tf.random.uniform((), minval=0, maxval=axis_len - width, dtype=tf.int32)

    if axis == 0:
        s1, s2, s3 = (start, shape[1], 3), (width, shape[1], 3), (axis_len - start - width, shape[1], 3)
    else:
        s1, s2, s3 = (shape[0], start, 3), (shape[0], width, 3), (shape[0], axis_len - start - width, 3)

    keep_mask = tf.concat([tf.ones(s1), tf.zeros(s2), tf.ones(s3)], axis=axis)
    return image * keep_mask + (1.0 - keep_mask) * fillValue


def specAugment(image, label):
    image = tf.cast(image, tf.float32)
    image = maskRandomSlice(image, axis=0)
    image = maskRandomSlice(image, axis=1)
    return image, label


class_weight_dict = dict(
    enumerate(
        np.power(
            compute_class_weight("balanced", classes=np.unique(y_tr_cnn), y=y_tr_cnn),
            0.5,
        )
    )
)

print(f"Sub-train: {len(X_tr_cnn)} | Val: {len(X_val_cnn)} | Test: {len(X_test_cnn)}")
print(f"Devices normalized: {list(device_stats.keys())}")
print(f"Pixel range: [{X_tr_cnn.min():.1f}, {X_tr_cnn.max():.1f}]")
print("Class weights:", class_weight_dict)

Sub-train: 3790 | Val: 352 | Test: 2756
Devices normalized: ['AKGC417L', 'Litt3200', 'LittC2SE', 'Meditron']
Pixel range: [0.0, 255.0]
Class weights: {0: 0.7322701558926816, 1: 0.9014472830491188, 2: 1.3849246168409253, 3: 1.615609122735312}


In [19]:
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

NUM_CLASSES = 4
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def mixup_batch(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    # Sample Beta(alpha, alpha) using two Gamma draws
    g1 = tf.random.gamma([batch_size, 1, 1, 1], alpha)
    g2 = tf.random.gamma([batch_size, 1, 1, 1], alpha)
    lam = g1 / (g1 + g2 + 1e-7)
    lam = tf.maximum(lam, 1.0 - lam)

    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)

    lam_y = tf.reshape(lam, [batch_size, 1])
    mixed_labels = lam_y * labels + (1.0 - lam_y) * tf.gather(labels, indices)
    return mixed_images, mixed_labels


def buildDataset(X, y, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, tf.one_hot(y, depth=NUM_CLASSES)))
    if is_training:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
        ds = ds.map(specAugment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(BATCH_SIZE, drop_remainder=True)
        ds = ds.map(mixup_batch, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)


train_ds_cnn = buildDataset(X_tr_cnn, y_tr_cnn, is_training=True)
val_ds_cnn   = buildDataset(X_val_cnn, y_val_cnn, is_training=False)
test_ds_cnn  = buildDataset(X_test_cnn, y_test_cnn, is_training=False)

input_shape = X_tr_cnn.shape[1:]  # (128, 256, 3)

base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape,
)
base_model.trainable = False

inputs = layers.Input(shape=input_shape)

x = resnet_preprocess(inputs)
feat = base_model(x, training=False)

pooled = layers.Concatenate()([
    layers.GlobalAveragePooling2D()(feat),
    layers.GlobalMaxPooling2D()(feat),
])

head = tf.keras.Sequential([
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(3e-4)),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="classifier_head")

full_model = tf.keras.Model(inputs, head(pooled), name="ResNet50_ICBHI")
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)

soft_class_weights = dict(
    enumerate(
        np.power(
            compute_class_weight("balanced", classes=np.unique(y_tr_cnn), y=y_tr_cnn),
            0.35,
        )
    )
)

cnn_start = time.time()

full_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss_fn,
    metrics=["accuracy"],
)
print("======== Stage 1: Training Classification Head ========")
history_stage1 = full_model.fit(
    train_ds_cnn,
    validation_data=val_ds_cnn,
    epochs=12,
    class_weight=soft_class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        )
    ],
)

base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

full_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss=loss_fn,
    metrics=["accuracy"],
)
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=7, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6
    ),
]
print("\n======== Stage 2: Fine-Tuning Top ResNet50 Block ========")
history_stage2 = full_model.fit(
    train_ds_cnn,
    validation_data=val_ds_cnn,
    epochs=25,
    class_weight=soft_class_weights,
    callbacks=callbacks,
)

cnn_train_time = time.time() - cnn_start
print(f"\nTraining complete in {cnn_train_time:.1f}s")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step
======== Stage 1: Training Classification Head ========
Epoch 1/12


/Users/henryburbridge/Desktop/AI-module-ARU/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


118/118 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.3705 - loss: 4.4635 - val_accuracy: 0.4744 - val_loss: 2.0376
Epoch 2/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.4388 - loss: 3.8630 - val_accuracy: 0.3778 - val_loss: 1.9659
Epoch 3/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.4449 - loss: 2.6452 - val_accuracy: 0.6165 - val_loss: 1.3219
Epoch 4/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 8s 64ms/step - accuracy: 0.4563 - loss: 1.9102 - val_accuracy: 0.6364 - val_loss: 1.1359
Epoch 5/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 8s 63ms/step - accuracy: 0.4785 - loss: 1.6022 - val_accuracy: 0.5653 - val_loss: 1.1325
Epoch 6/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.5042 - loss: 1.4108 - val_accuracy: 0.6676 - val_loss: 1.0920
Epoch 7/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.5164 - loss: 1.3675 - val_accuracy: 0.6477 - val_loss: 1.0508
Epoch 8/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.5289 - loss: 1.3279 - val_accuracy: 0.62

In [22]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Generate class predictions from the trained CNN
y_pred_cnn = np.argmax(full_model.predict(test_ds_cnn), axis=1)
predictions["CNN"] = y_pred_cnn

# Calculate CNN training time in minutes and seconds
mins, secs = divmod(int(cnn_train_time), 60)

acc = accuracy_score(y_test_cnn, y_pred_cnn)
prec, rec, f1, _ = precision_recall_fscore_support(y_test_cnn, y_pred_cnn, average="macro", zero_division=0)
se, sp, icbhi = icbhiScore(y_test_cnn, y_pred_cnn)

resultsCNN = pd.DataFrame([{
    "Model": "CNN",
    "Accuracy": acc,
    "ICBHI Score": icbhi,
    "Se": se,
    "Sp": sp,
    "Precision (Macro)": prec,
    "Recall (Macro)": rec,
    "F1-Score (Macro)": f1,
    "Training Time": f"{mins}m {secs}s",
}])

results_df = (
    pd.concat([results_df, resultsCNN], ignore_index=True)
    .drop_duplicates(subset=["Model"], keep="last")
    .sort_values(by="ICBHI Score", ascending=False)
    .reset_index(drop=True)
)

print("============================== Model Comparison ==============================\n")
print(results_df.to_string(index=False))

# ---------------------------------------------------------
# Confusion Matrices (Sorted by ICBHI Score)
# ---------------------------------------------------------
CLASS_NAMES = ["Normal", "Crackle", "Wheeze", "Both"]
ranked_models = results_df["Model"].tolist()

print("\n============================== Confusion Matrices ==============================")
for name in ranked_models:
    y_true_model = y_test_cnn if name == "CNN" else y_test
    cm = confusion_matrix(y_true_model, predictions[name], labels=[0, 1, 2, 3])
    cm_df = pd.DataFrame(cm, index=[f"True_{c}" for c in CLASS_NAMES], columns=[f"Pred_{c}" for c in CLASS_NAMES])
    print(f"\n--- {name} (ICBHI Score: {results_df.loc[results_df['Model'] == name, 'ICBHI Score'].values[0]:.4f}) ---")
    print(cm_df)

# Plot a 3x3 Grid of All Confusion Matrices
n_models = len(ranked_models)
cols = 3
rows = int(np.ceil(n_models / cols))

fig, axes = plt.subplots(rows, cols, figsize=(16, 4.8 * rows))
axes = axes.flatten()

for idx, name in enumerate(ranked_models):
    y_true_model = y_test_cnn if name == "CNN" else y_test
    cm = confusion_matrix(y_true_model, predictions[name], labels=[0, 1, 2, 3])

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=axes[idx], cmap="Blues", colorbar=False, values_format="d")

    row_metrics = results_df.loc[results_df["Model"] == name].iloc[0]
    axes[idx].set_title(
        f"{name}\nICBHI: {row_metrics['ICBHI Score']:.3f} | Acc: {row_metrics['Accuracy']:.3f} | F1: {row_metrics['F1-Score (Macro)']:.3f}",
        fontsize=11,
        fontweight="bold",
    )
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("True Label")

# Hide any unused subplot slots if total models != multiple of 3
for j in range(n_models, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

87/87 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step


NameError: name 'predictions' is not defined